In [4]:
import os
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [5]:
def create_mfcc_features(clean_dir, n_mfcc=13, hop_length=512):
    clean_dir = Path(clean_dir)
    X = []
    y = []
    labels = [f.name for f in clean_dir.iterdir() if f.is_dir()]

    for label in tqdm(labels):
        for audio in (clean_dir / label).iterdir():
            if audio.suffix == ".wav":
                signal, sr = librosa.load(audio, sr=None)
                n_fft = min(2048, len(signal))
                mfcc = librosa.feature.mfcc(y=signal, 
                                            sr=sr, 
                                            n_mfcc=n_mfcc, 
                                            n_fft=n_fft, 
                                            hop_length=hop_length)
                mfcc_mean = np.mean(mfcc, axis=1)
                mfcc_std = np.std(mfcc, axis=1)
                mfcc_features = np.concatenate([mfcc_mean, mfcc_std])
                X.append(mfcc_features)
                y.append(label)

    return np.array(X), np.array(y)

Load data

In [6]:
X, y = create_mfcc_features('clean')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=1)

100%|██████████| 9/9 [00:38<00:00,  4.32s/it]


Train SVM

In [7]:
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)

SVC(C=1.0, break_ties=False, cache_size=200, class_weight=None, coef0=0.0,
    decision_function_shape='ovr', degree=3, gamma='scale', kernel='linear',
    max_iter=-1, probability=False, random_state=None, shrinking=True,
    tol=0.001, verbose=False)

Evaluate SVM

In [8]:
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print('Accuracy:', accuracy)
print('\nClassification Report:\n', class_report)

output_file = 'svm_evaluation.txt'

# with open(output_file, 'w') as file:
#     file.write(f'Accuracy: {accuracy}\n')
#     file.write('\nClassification Report:\n')
#     file.write(class_report)

# best .734

Accuracy: 0.49290444654683063

Classification Report:
               precision    recall  f1-score   support

         cel       0.55      0.60      0.58        78
         cla       0.51      0.40      0.45       101
         flu       0.48      0.34      0.40        90
         gac       0.47      0.50      0.49       127
         gel       0.44      0.56      0.49       152
         org       0.49      0.65      0.56       137
         pia       0.54      0.60      0.57       144
         sax       0.41      0.27      0.33       125
         tru       0.59      0.44      0.50       103

    accuracy                           0.49      1057
   macro avg       0.50      0.48      0.48      1057
weighted avg       0.49      0.49      0.49      1057



Save SVM

In [9]:
# model_filepath = 'svm_model.pkl'

# with open(model_filepath, 'wb') as model_file:
#     pickle.dump(svm, model_file)

# print(f'SVM saved to {os.path.abspath(output_file)}')